# Keyword Labels vs. Structured Benefit Categories

This notebook compares our keyword-based benefit labels (from `label_benefits.py` / `label_benefits_remote.py`) against the structured `BENEFIT_NAME`, `BENEFIT_SUBCATEGORY_NAME`, and `BENEFIT_CATEGORIES_NAME` columns in the original data.

**Goal:** Assess how well the keyword labels align with the structured fields — identify where they agree, where they diverge, and what each approach captures that the other misses.

In [18]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_rows", 60)

base = Path("..") / "data" / "processed"
data = pd.read_parquet(base / "labeled_v1.parquet")
print(f"Rows: {data.shape[0]:,}")

# Keyword labels
kw_labels = ["EDU_ASSISTANCE", "PAID LEAVE", "HEALTH_WELLBEING", "PARENTAL_LEAVE", "CULTURE", "REMOTE_KW", "WELLBEING"]

# Structured columns are stored as JSON-style string arrays — parse them
for col in ["BENEFIT_NAME", "BENEFIT_SUBCATEGORY_NAME", "BENEFIT_CATEGORIES_NAME"]:
    data[col] = data[col].apply(lambda x: json.loads(x) if isinstance(x, str) and x.startswith("[") else [])

print(f"Rows with any structured benefit: {data['BENEFIT_NAME'].apply(len).gt(0).sum():,}")
print(f"Rows with any keyword label: {data[kw_labels].any(axis=1).sum():,}")

Rows: 99,860
Rows with any structured benefit: 49,359
Rows with any keyword label: 41,703


## 1. Structured benefit field coverage

First, let's see what unique values appear in each structured column and how often.

In [19]:
# Explode each structured column to get individual values
for col in ["BENEFIT_CATEGORIES_NAME", "BENEFIT_SUBCATEGORY_NAME", "BENEFIT_NAME"]:
    exploded = data[col].explode().dropna()
    exploded = exploded[exploded != ""]
    print(f"\n=== {col} — {exploded.nunique()} unique values ===")
    display(exploded.value_counts().head(25).to_frame("count"))


=== BENEFIT_CATEGORIES_NAME — 8 unique values ===


,count
BENEFIT_CATEGORIES_NAME,
Insurance,83358
Paid Leave,36375
Retirement and Savings,35605
Education and Career Development,17070
Work-Life Balance,12983
Supplemental Pay,11856
Other Benefits,8700
Health and Wellness Benefits,8691



=== BENEFIT_SUBCATEGORY_NAME — 44 unique values ===


,count
BENEFIT_SUBCATEGORY_NAME,
401(k) Plans,26571
Paid Time Off (PTO),25902
Dental Insurance,21923
Vision Insurance,21117
Health Insurance,19117
Life Insurance,14336
Flexible Work Schedules,9816
Financial Aid/Assistance,8457
Other Retirement and Savings,8253



=== BENEFIT_NAME — 44 unique values ===


,count
BENEFIT_NAME,
401(k) Plans,26571
Paid Time Off (PTO),25902
Dental Insurance,21923
Vision Insurance,21117
Health Insurance,19117
Life Insurance,14336
Flexible Work Schedules,9816
Financial Aid/Assistance,8457
Other Retirement and Savings,8253


In [20]:
# Mapping: keyword label -> structured BENEFIT_SUBCATEGORY_NAME values
# CULTURE and REMOTE_KW have no structured equivalent
keyword_to_structured = {
    "HEALTH_WELLBEING": ["Health and Wellness Programs"],
    "PARENTAL_LEAVE": ["Parental Leave"],
    "PAID LEAVE": ["Sick Leave", "Floating Holidays", "Paid Time Off (PTO)"],
    "EDU_ASSISTANCE": ["Financial Aid/Assistance"],
}

# Build structured indicators from the mapping
def has_any(series, terms):
    """Check if any of `terms` appear in the list-valued column."""
    terms_lower = [t.lower() for t in terms]
    return series.apply(lambda lst: any(v.lower() in terms_lower for v in lst if isinstance(v, str)))

for kw_col, struct_vals in keyword_to_structured.items():
    col_name = f"S_{kw_col}"
    data[col_name] = has_any(data["BENEFIT_SUBCATEGORY_NAME"], struct_vals)
    print(f"{col_name}: {data[col_name].sum():,} rows  (from {struct_vals})")

S_HEALTH_WELLBEING: 8,220 rows  (from ['Health and Wellness Programs'])
S_PARENTAL_LEAVE: 4,490 rows  (from ['Parental Leave'])
S_PAID LEAVE: 27,275 rows  (from ['Sick Leave', 'Floating Holidays', 'Paid Time Off (PTO)'])
S_EDU_ASSISTANCE: 8,457 rows  (from ['Financial Aid/Assistance'])


## 2. Which structured benefit values co-occur with each keyword label?

Rather than pre-assuming mappings, let the data show which structured categories, subcategories, and benefit names appear most often among rows flagged by each keyword label.

In [21]:
kw_to_label = {
    "EDU_ASSISTANCE": "Tuition Assistance",
    "PAID LEAVE": "Paid Leave",
    "HEALTH_WELLBEING": "Health & Wellbeing",
    "PARENTAL_LEAVE": "Parental Leave",
    "CULTURE": "Workplace Culture",
    "REMOTE_KW": "Remote Work",
}

for kw_col, label in kw_to_label.items():
    flagged = data[data[kw_col].astype(bool)]
    print(f"\n{'='*70}")
    print(f"{label} ({kw_col}) — {len(flagged):,} rows flagged by keyword")
    print(f"{'='*70}")

    for struct_col in ["BENEFIT_CATEGORIES_NAME", "BENEFIT_SUBCATEGORY_NAME", "BENEFIT_NAME"]:
        exploded = flagged[struct_col].explode().dropna()
        exploded = exploded[exploded != ""]
        if len(exploded) == 0:
            print(f"\n  {struct_col}: no values")
            continue
        top = exploded.value_counts().head(10)
        total_flagged = len(flagged)
        print(f"\n  {struct_col} (top 10):")
        for val, count in top.items():
            print(f"    {val:45s}  {count:>6,}  ({count/total_flagged*100:5.1f}%)")



Tuition Assistance (EDU_ASSISTANCE) — 9,273 rows flagged by keyword

  BENEFIT_CATEGORIES_NAME (top 10):
    Insurance                                      20,427  (220.3%)
    Education and Career Development               10,724  (115.6%)
    Paid Leave                                     10,007  (107.9%)
    Retirement and Savings                          9,251  ( 99.8%)
    Supplemental Pay                                3,093  ( 33.4%)
    Health and Wellness Benefits                    2,786  ( 30.0%)
    Other Benefits                                  2,459  ( 26.5%)
    Work-Life Balance                               2,030  ( 21.9%)

  BENEFIT_SUBCATEGORY_NAME (top 10):
    Financial Aid/Assistance                        8,286  ( 89.4%)
    401(k) Plans                                    6,661  ( 71.8%)
    Paid Time Off (PTO)                             6,495  ( 70.0%)
    Vision Insurance                                5,300  ( 57.2%)
    Dental Insurance                    

## 3. Cross-tabulation: Keyword vs. Structured

For each mapped benefit, compare the keyword label against the structured indicator. CULTURE and REMOTE_KW have no structured counterpart and are excluded.

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

benefit_names = {
    "HEALTH_WELLBEING": "Health & Wellbeing",
    "PARENTAL_LEAVE": "Parental Leave",
    "PAID LEAVE": "Paid Leave",
    "EDU_ASSISTANCE": "Tuition Assistance",
}

summary_rows = []
for kw_col in keyword_to_structured:
    struct_col = f"S_{kw_col}"
    kw = data[kw_col].astype(bool)
    st = data[struct_col].astype(bool)

    tn, fp, fn, tp = confusion_matrix(st, kw).ravel()
    prec = precision_score(st, kw, zero_division=0)
    rec  = recall_score(st, kw, zero_division=0)
    f1   = f1_score(st, kw, zero_division=0)

    summary_rows.append({
        "Benefit": benefit_names[kw_col],
        "Keyword+": int(kw.sum()),
        "Structured+": int(st.sum()),
        "Both+": int(tp),
        "KW only": int(fp),
        "Struct only": int(fn),
        "Precision": round(prec, 3),
        "Recall": round(rec, 3),
        "F1": round(f1, 3),
    })

summary = pd.DataFrame(summary_rows)
display(summary)

## 4. Detailed cross-tabs per benefit

In [ ]:
for kw_col in keyword_to_structured:
    struct_col = f"S_{kw_col}"
    name = benefit_names[kw_col]
    kw = data[kw_col].astype(bool)
    st = data[struct_col].astype(bool)

    ct = pd.crosstab(st, kw, rownames=[f"Structured ({struct_col})"], colnames=[f"Keyword ({kw_col})"])
    print(f"\n{'='*60}")
    print(f"{name}  —  Structured = {keyword_to_structured[kw_col]}")
    print(f"{'='*60}")
    display(ct)

## 5. Inspect disagreements

For each benefit, sample rows where the keyword and structured labels disagree to understand *why* they diverge.

In [ ]:
sample_n = 5

for kw_col in keyword_to_structured:
    struct_col = f"S_{kw_col}"
    name = benefit_names[kw_col]
    kw = data[kw_col].astype(bool)
    st = data[struct_col].astype(bool)

    kw_only = data[kw & ~st]
    st_only = data[~kw & st]

    print(f"\n{'='*60}")
    print(f"{name}")
    print(f"{'='*60}")

    if len(kw_only) > 0:
        print(f"\n--- Keyword+ / Structured- ({len(kw_only):,} rows) — sample {min(sample_n, len(kw_only))} ---")
        sample = kw_only.sample(min(sample_n, len(kw_only)), random_state=42)
        for _, row in sample.iterrows():
            print(f"  BENEFIT_NAME: {row['BENEFIT_NAME']}")
            print(f"  BENEFIT_SUBCATEGORY: {row['BENEFIT_SUBCATEGORY_NAME']}")
            body = str(row.get("BODY", ""))[:200]
            if body:
                print(f"  BODY snippet: {body}...")
            print()
    else:
        print(f"\n--- No Keyword+ / Structured- cases ---")

    if len(st_only) > 0:
        print(f"--- Structured+ / Keyword- ({len(st_only):,} rows) — sample {min(sample_n, len(st_only))} ---")
        sample = st_only.sample(min(sample_n, len(st_only)), random_state=42)
        for _, row in sample.iterrows():
            print(f"  BENEFIT_NAME: {row['BENEFIT_NAME']}")
            print(f"  BENEFIT_SUBCATEGORY: {row['BENEFIT_SUBCATEGORY_NAME']}")
            body = str(row.get("BODY", ""))[:200]
            if body:
                print(f"  BODY snippet: {body}...")
            print()
    else:
        print(f"--- No Structured+ / Keyword- cases ---")

## 6. Coverage comparison: structured field completeness

The structured benefit fields may be sparsely populated (many rows have `[]`). Compare what fraction of postings have *any* structured benefit vs. *any* keyword-detected benefit, broken down by year.

In [16]:
data["has_structured"] = data["BENEFIT_NAME"].apply(len).gt(0)
data["has_keyword"] = data[kw_labels].any(axis=1)

coverage = data.groupby("YEAR").agg(
    n=("ID", "size"),
    pct_structured=("has_structured", "mean"),
    pct_keyword=("has_keyword", "mean"),
).round(4)
coverage["pct_structured"] = (coverage["pct_structured"] * 100).round(1)
coverage["pct_keyword"] = (coverage["pct_keyword"] * 100).round(1)
display(coverage)

,n,pct_structured,pct_keyword
YEAR,,,
2018,10199,29.4,21.2
2019,10789,34.5,25.6
2020,10615,42.3,33.0
2021,13999,48.4,40.2
2022,15529,51.7,45.7
2023,13274,56.7,50.3
2024,12304,61.7,55.2
2025,13151,62.5,53.8


## 7. Prevalence comparison by AI ROLE

Do the two approaches tell the same story about AI vs non-AI differences?

In [ ]:
rows = []
for kw_col in keyword_to_structured:
    struct_col = f"S_{kw_col}"
    name = benefit_names[kw_col]
    for label, source in [(kw_col, "Keyword"), (struct_col, "Structured")]:
        ai = data[data["AI ROLE"]][label].mean()
        non_ai = data[~data["AI ROLE"]][label].mean()
        rows.append({
            "Benefit": name,
            "Source": source,
            "AI (%)": round(ai * 100, 2),
            "Non-AI (%)": round(non_ai * 100, 2),
            "Diff (pp)": round((ai - non_ai) * 100, 2),
        })

comparison = pd.DataFrame(rows)
display(comparison.pivot_table(index="Benefit", columns="Source", values=["AI (%)", "Non-AI (%)", "Diff (pp)"]))